# genEPJ Windows tiny-home smoke demo

This notebook verifies a fresh genEPJ installation using the packaged tiny-home example. It is intentionally lightweight and does not require EnergyPlus unless the optional final check is run.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
print('Python:', sys.version)
print('Working directory:', ROOT)

## 1. Import genEPJ and locate the packaged epJSON template

In [ ]:
import genEPJ
from genEPJ.resources import epjson_template_path

template = epjson_template_path()
print('Template path:', template)
print('Template exists:', template.is_file())

## 2. Locate the tiny-home demonstration files

In [ ]:
demo = ROOT / 'examples' / 'sim-tinyhome'
idf = demo / 'nomad_3RV96.idf'
workflow = demo / 'Nomad_genEPJ.py'
inputs = demo / 'opti_inputs.json'

for path in [idf, workflow, inputs]:
    print(path, 'exists:', path.is_file())

## 3. Parse the IDF file with genEPJ

In [ ]:
from genEPJ.generate_EPJ import get_IDF_objs_raw
from genEPJ.pylib.genEPJ_lib import get_obj_type

objects = get_IDF_objs_raw(str(idf))
object_types = {get_obj_type(obj) for obj in objects}
print('Number of parsed EnergyPlus objects:', len(objects))
print('Contains Version:', 'Version' in object_types)
print('Contains Building:', 'Building' in object_types)
print('Contains BuildingSurface:Detailed:', 'BuildingSurface:Detailed' in object_types)

## 4. Run the Windows-safe check_model smoke command

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'genEPJ.standalone.check_model', str(idf)],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
print('Return code:', result.returncode)

## 5. Optional: check whether EnergyPlus is available on PATH

In [ ]:
import shutil

energyplus = shutil.which('energyplus')
print('energyplus executable:', energyplus)
if energyplus:
    version = subprocess.run(['energyplus', '--version'], capture_output=True, text=True)
    print(version.stdout or version.stderr)
else:
    print('EnergyPlus is not on PATH. This is acceptable for the smoke test.')